In [1]:
import numpy as np
import pandas as pd
from scipy import stats

# Завдання 1. Генерація вибірки денних температур для Варіанта 2 (Львів, липень = 18 °C)
np.random.seed(2)  # фіксуємо seed номером варіанта

daily_temps = np.random.normal(loc=18.0, scale=2.5, size=30)

print("Згенеровані денні температури (°C):")
print(np.round(daily_temps, 2))

mean_val = daily_temps.mean()
s_val = daily_temps.std(ddof=1)  # вибіркове стандартне відхилення (оцінка справжнього сигналу σ)
n = len(daily_temps)

print(f"\nВибіркове середнє (mean): {mean_val:.2f} °C")
print(f"Вибіркове стандартне відхилення (s): {s_val:.2f} °C")
print(f"Обсяг вибірки (n): {n}")

Згенеровані денні температури (°C):
[16.96 17.86 12.66 22.1  13.52 15.9  19.26 14.89 15.36 15.73 19.38 23.73
 18.1  15.21 19.35 16.51 17.95 20.94 16.13 18.02 15.8  17.61 18.64 15.53
 17.15 17.41 16.41 15.03 14.45 17.62]

Вибіркове середнє (mean): 17.17 °C
Вибіркове стандартне відхилення (s): 2.42 °C
Обсяг вибірки (n): 30


In [2]:
# Завдання 2. Побудова 95% довірчого інтервалу для середнього через t-розподіл
se = s_val / np.sqrt(n)

# stats.t.interval приймає рівень довіри (0.95), кількість ступенів свободи (df = n - 1), середнє та стандартну похибку
ci_mean_95 = stats.t.interval(0.95, df=n - 1, loc=mean_val, scale=se)

print(f"Стандартна похибка (se): {se:.4f}")
print(f"95% довірчий інтервал для середньої температури: ({ci_mean_95[0]:.2f}, {ci_mean_95[1]:.2f}) °C")

Стандартна похибка (se): 0.4411
95% довірчий інтервал для середньої температури: (16.27, 18.07) °C


### Відповідь на Завдання 2. Чому використовується t-розподіл, а не z-критичне?
* **Обґрунтування:** Справжнє стандартне відхилення генеральної сукупності ($\sigma$) клімату міста невідоме; ми маємо лише його вибіркову оцінку $s$, обчислену за малою вибіркою ($n = 30$). Оскільки ми замінюємо невідому $\sigma$ на вибіркову оцінку $s$, статистика підпорядковується не нормальному $z$-розподілу, а $t$-розподілу Стьюдента із $n - 1$ ступенями свободи, який має ширші хвости і враховує додаткову невизначеність.

In [3]:
# Завдання 3. Довірчий інтервал для частки днів із температурою вище липневої норми (18 °C)
table_value = 18.0
under_condition = daily_temps > table_value

p_hat = under_condition.mean()
n_success = n * p_hat
n_failure = n * (1 - p_hat)

print(f"Частка днів вище 18 °C (p_hat): {p_hat:.4f}")
print(f"Кількість успіхів (n * p_hat): {n_success:.1f}")
print(f"Кількість невдач (n * (1 - p_hat)): {n_failure:.1f}")

# Перевірка умови коректності нормального наближення (обидва значення мають бути ≥ 5)
if n_success >= 5 and n_failure >= 5:
    print("Умова нормального наближення виконана (≥ 5).")
else:
    print("Умова не виконується повністю, наближення може бути менш точним.")

# Побудова 95% довірчого інтервалу для частки
se_p = np.sqrt(p_hat * (1 - p_hat) / n)
ci_prop_95 = stats.norm.interval(0.95, loc=p_hat, scale=se_p)

print(f"95% довірчий інтервал для частки: ({ci_prop_95[0]:.4f}, {ci_prop_95[1]:.4f})")

Частка днів вище 18 °C (p_hat): 0.3000
Кількість успіхів (n * p_hat): 9.0
Кількість невдач (n * (1 - p_hat)): 21.0
Умова нормального наближення виконана (≥ 5).
95% довірчий інтервал для частки: (0.1360, 0.4640)


### Відповідь на Завдання 3. Аналіз частки та перевірка теоретичного значення
* **Перевірка умови:** Значення $n \cdot \hat{p}$ та $n \cdot (1 - \hat{p})$ є достатніми (обидва $\ge 5$), що дозволяє безпечно використовувати нормальне наближення.
* **Чи потрапляє 0.5 у довірчий інтервал:** Теоретично вибірка генерувалася симетрично навколо 18 °C, тому істинна частка мала б дорівнювати 0.5. Отриманий інтервал може як містити 0.5, так і дещо зміщуватися в бік залежно від конкретної випадкової реалізації вибірки. Якщо 0.5 не потрапило в інтервал, це є нормальним статистичним явищем: близько 5% випадкових вибірок дають інтервали, які не накривають істинне значення (саме в цьому і полягає зміст рівня довіри 95%).

In [4]:
# Завдання 4. Порівняння ширини інтервалу для рівнів довіри 90%, 95% і 99%
confidence_levels = [0.90, 0.95, 0.99]
results_ci = []

for cl in confidence_levels:
    ci = stats.t.interval(cl, df=n - 1, loc=mean_val, scale=se)
    width = ci[1] - ci[0]
    results_ci.append({
        "Рівень довіри": f"{int(cl * 100)}%", 
        "Нижня межа": ci[0], 
        "Верхня межа": ci[1], 
        "Ширина інтервалу": width
    })

df_ci_widths = pd.DataFrame(results_ci)
for r in results_ci:
    print(f"Рівень довіри: {r['Рівень довіри']} | Інтервал: ({r['Нижня межа']:.2f}, {r['Верхня межа']:.2f}) | Ширина: {r['Ширина інтервалу']:.4f}")
 

Рівень довіри: 90% | Інтервал: (16.42, 17.92) | Ширина: 1.4990
Рівень довіри: 95% | Інтервал: (16.27, 18.07) | Ширина: 1.8044
Рівень довіри: 99% | Інтервал: (15.96, 18.39) | Ширина: 2.4318


### Відповідь на Завдання 4. Зростання ширини та «плата» за довіру
* **Чому ширина зростає:** Зі збільшенням рівня довіри (від 90% до 99%) критичне значення t-статистики зростає, оскільки ми вимагаємо більшої гарантії накриття істинного параметра.
* **Яку «плату» доводиться платити:** Щоб отримати вищий рівень довіри (наприклад, 99%), доводиться «платити» **більшою шириною інтервалу** (інтервал стає менш точним, розмитішим). Щоб зробити інтервал вузьким і водночас високо надійним, єдиний спосіб — збільшити розмір вибірки ($n$).

### Відповідь на Завдання 5. Коректна інтерпретація 95% довірчого інтервалу
* **Змістовне пояснення:** Отриманий 95% довірчий інтервал означає не те, що істинна середня температура лежить у ньому з 95% імовірністю, а те, що **процедура побудови** такого інтервалу є надійною в довгостроковій перспективі. Тобто, якщо повторити експеримент 100 разів (згенерувати 100 різних вибірок по 30 днів і для кожної побудувати свій 95% довірчий інтервал), приблизно в 95 випадках із 100 побудований інтервал успішно накриє справжнє невідоме середнє значення генеральної сукупності. Для одного вже обчисленого інтервалу параметр або належить йому повністю (з імовірністю 1), або не належить зовсім (0).

---

### Відповіді на контрольні запитання

1. **Різниця між точковою оцінкою і довірчим інтервалом:**
   Точкова оцінка (наприклад, вибіркове середнє $\bar{x}$) дає лише одне конкретне число, яке майже ніколи не збігається з істинним параметром через випадкову похибку вибірки. Довірчий інтервал додає до точкової оцінки діапазон можливих значень разом із мірою надійності (рівнем довіри), що набагато інформативніше.

2. **Чому використовують t-розподіл замість нормального при невідомому $\sigma$ і малому $n$:**
   Оскільки стандартне відхилення генеральної сукупності невідоме, його заміна вибірковим оціночним $s$ вносить додаткову статистичну похибку. t-розподіл Стьюдента має важчі хвости, що компенсує цю невизначеність для малих вибірок.

3. **Що насправді означає «95% довіри»:**
   Це частотна характеристика методу: у довгостроковій серії випробувань 95% побудованих таким чином інтервалів міститимуть істинне значення. Помилково говорити про «95% імовірності для конкретного інтервалу», оскільки параметр не є випадковою величиною — він фіксований, а випадковим є сам інтервал від вибірки до вибірки.

4. **Три фактори, що визначають ширину довірчого інтервалу:**
   * **Рівень довіри ($1 - \alpha$):** вищий рівень $\rightarrow$ ширший інтервал.
   * **Вибіркове стандартне відхилення ($s$):** більша мінливість даних $\rightarrow$ ширший інтервал.
   * **Розмір вибірки ($n$):** більший обсяг вибірки $\rightarrow$ менша стандартна похибка $\rightarrow$ вужчий інтервал.